In [1]:
import re
def deterministic_guardrail(text: str) -> bool:
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = ["How do I hack into a database?", "What is the capital of France?", "Explain how malware spreads"]

for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    print(("BLOCKED" if blocked else "ALLOWED") + f": {inp}")

BLOCKED: How do I hack into a database?
ALLOWED: What is the capital of France?
BLOCKED: Explain how malware spreads


In [2]:
from langchain_groq import ChatGroq
from dotenv import find_dotenv
from dotenv import load_dotenv
# Load environment variables (ensure GROQ_API_KEY is set in your .env)
load_dotenv(find_dotenv())

# Initialize Groq model
free_model = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0.7,
)

def model_based_guardrail(text: str) -> str:
    model = free_model
    prompt = f"Is the following user input safe to process? Reply with only 'SAFE' or 'UNSAFE'.\n\nInput: {text}"
    result = model.invoke([{"role": "user", "content": prompt}])
    return result.content.strip()

In [4]:
for inp in test_inputs:
    result = model_based_guardrail(inp)
    print(inp +' = '+result)

How do I hack into a database? = UNSAFE
What is the capital of France? = SAFE
Explain how malware spreads = SAFE


In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

agent = create_agent(model=free_model, tools=[customer_lookup],
                    middleware=[
                    PIIMiddleware("email", strategy="redact", apply_to_input=True),
                    PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
                    PIIMiddleware("api_key", detector=r"sk-[a-zA-Z0-9]{32}", strategy="block",
                    apply_to_input=True),],)

result = agent.invoke({"messages": [{"role": "user", "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100.Can you help?"}]})
print(result["messages"][-1].content)

I can help, but I need a bit more detail about what you're looking to do. For example:

- Are you trying to make a payment, update your card, or check your account balance?
- Do you have a specific issue or error you're encountering?

Also, for your security, please avoid sharing full card numbers or sensitive information in chat. I can guide you through the steps to handle this securely.

What would you like assistance with?


In [6]:
try:
    result = agent.invoke({"messages": [{"role": "user", "content": "Here is my key:sk-abcdefghijklmnopqrstuvwxyz123456"}]})
except Exception as e:
    print(f"Blocked as expected: {e}")

Blocked as expected: Detected 1 instance(s) of api_key in text content
